In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.metrics import confusion_matrix

In [2]:
class SingleLayer(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Linear(input_size, 8),
            nn.ReLU(),
            #nn.Dropout(0.25), # works slightly better without the dropout layer, and considering we're already doing mini-batching and a train-test split, I'm not too concerned about overfitting
            nn.Linear(8, 2))
    def forward(self, x):
        return(self.sequential(x))

In [3]:
mystery_data = pd.read_csv("../../data/FP_Data.csv")

mystery_data_onehot = pd.get_dummies(mystery_data)

y = mystery_data_onehot.pop("y")
y = np.array([1 if obs >= 70 else 0 for obs in y])
X = mystery_data_onehot

X = normalize(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.75, random_state=28)

In [4]:
model = SingleLayer(11)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.1) # This is, as far as I seen, the most widely used optimizer, though it is not what is used in the textbook
loss_fn = nn.CrossEntropyLoss()
#log_softmax = nn.functional.log_softmax(dim = 1, dtype = torch.float32)

epochs = 50
batch_size = 32

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(np.asarray(y_train), dtype=torch.long)
y_test = torch.tensor(np.asarray(y_test), dtype=torch.long)

for epoch in range(epochs):
    model.train()

    permutation = torch.randperm(X_train.size(0))
    for i in range(0, X_train.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        X_batch, y_batch = X_train[indices], y_train[indices]

        optimizer.zero_grad()
        output = model(X_batch)
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_output = model(X_test)
        val_loss = loss_fn(val_output, y_test)
    print(f"Epoch {epoch+1}/{epochs} | Val Loss: {val_loss.item():.4f}")

Epoch 1/50 | Val Loss: 0.4031
Epoch 2/50 | Val Loss: 0.2800
Epoch 3/50 | Val Loss: 0.2654
Epoch 4/50 | Val Loss: 0.2677
Epoch 5/50 | Val Loss: 0.2735
Epoch 6/50 | Val Loss: 0.2920
Epoch 7/50 | Val Loss: 0.2796
Epoch 8/50 | Val Loss: 0.2988
Epoch 9/50 | Val Loss: 0.3116
Epoch 10/50 | Val Loss: 0.3183
Epoch 11/50 | Val Loss: 0.3019
Epoch 12/50 | Val Loss: 0.2860
Epoch 13/50 | Val Loss: 0.2907
Epoch 14/50 | Val Loss: 0.3041
Epoch 15/50 | Val Loss: 0.3248
Epoch 16/50 | Val Loss: 0.3527
Epoch 17/50 | Val Loss: 0.3308
Epoch 18/50 | Val Loss: 0.3581
Epoch 19/50 | Val Loss: 0.4054
Epoch 20/50 | Val Loss: 0.4124
Epoch 21/50 | Val Loss: 0.4312
Epoch 22/50 | Val Loss: 0.4494
Epoch 23/50 | Val Loss: 0.4431
Epoch 24/50 | Val Loss: 0.4658
Epoch 25/50 | Val Loss: 0.4966
Epoch 26/50 | Val Loss: 0.5319
Epoch 27/50 | Val Loss: 0.5138
Epoch 28/50 | Val Loss: 0.5545
Epoch 29/50 | Val Loss: 0.5693
Epoch 30/50 | Val Loss: 0.5373
Epoch 31/50 | Val Loss: 0.5787
Epoch 32/50 | Val Loss: 0.5943
Epoch 33/50 | Val

In [5]:
model.eval()
y_test = y_test.detach().numpy()
with torch.no_grad():
    test_pred = model(X_test).argmax(dim=1)
    test_acc = (test_pred == y_test).float().mean().item()
    confusion = confusion_matrix(y_test, test_pred)
    print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 0.8000
